# 📊 Statistics for Machine Learning

Welcome! Statistics is the science of learning from data - exactly what machine learning does!

## Why Statistics?

- **Hypothesis testing** - Is this model better than that one?
- **Confidence intervals** - How certain are we?
- **A/B testing** - Which version performs better?
- **Feature selection** - Which variables matter?

## What You'll Learn
1. Descriptive statistics
2. Statistical inference
3. Hypothesis testing
4. Confidence intervals
5. Correlation vs causation
6. Bias-variance tradeoff

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
import pandas as pd

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
np.random.seed(42)
%matplotlib inline

## 1. Descriptive Statistics

### Measures of Central Tendency
- **Mean**: Average value
- **Median**: Middle value
- **Mode**: Most common value

### Measures of Spread
- **Variance**: Average squared deviation from mean
- **Standard deviation**: Square root of variance
- **Quantiles**: 25th, 50th (median), 75th percentiles

In [ ]:
# Generate sample data
np.random.seed(42)
data = np.concatenate([np.random.normal(50, 10, 800), 
                       np.random.normal(80, 5, 200)])  # Bimodal

# Compute statistics
mean = np.mean(data)
median = np.median(data)
std = np.std(data)
q25, q75 = np.percentile(data, [25, 75])

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Histogram with statistics
ax1.hist(data, bins=50, density=True, alpha=0.7, 
         color='skyblue', edgecolor='black', linewidth=1)
ax1.axvline(mean, color='red', linestyle='--', linewidth=2.5, label=f'Mean = {mean:.1f}')
ax1.axvline(median, color='green', linestyle='--', linewidth=2.5, label=f'Median = {median:.1f}')
ax1.axvline(mean - std, color='orange', linestyle=':', linewidth=2, alpha=0.7)
ax1.axvline(mean + std, color='orange', linestyle=':', linewidth=2, alpha=0.7, label=f'±1 SD')
ax1.set_xlabel('Value', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('Distribution with Key Statistics', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Box plot
bp = ax2.boxplot(data, vert=True, patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][0].set_edgecolor('black')
bp['boxes'][0].set_linewidth(2)
for element in ['whiskers', 'fliers', 'means', 'medians', 'caps']:
    plt.setp(bp[element], color='black', linewidth=2)
plt.setp(bp['medians'], color='red', linewidth=3)

# Add annotations
ax2.text(1.3, median, f'Median\n{median:.1f}', fontsize=11, va='center')
ax2.text(1.3, q25, f'Q1\n{q25:.1f}', fontsize=11, va='center')
ax2.text(1.3, q75, f'Q3\n{q75:.1f}', fontsize=11, va='center')
ax2.set_ylabel('Value', fontsize=12)
ax2.set_title('Box Plot: Visualizing Quartiles', fontsize=13, fontweight='bold')
ax2.set_xticks([1])
ax2.set_xticklabels(['Data'])
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('📈 Descriptive Statistics', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print(f"Mean: {mean:.2f}")
print(f"Median: {median:.2f}")
print(f"Std Dev: {std:.2f}")
print(f"25th percentile: {q25:.2f}")
print(f"75th percentile: {q75:.2f}")
print(f"IQR (Interquartile Range): {q75 - q25:.2f}")

## 2. Hypothesis Testing

**Question**: Is the difference we observe real or just random chance?

### Process:
1. **Null hypothesis (H₀)**: No effect/difference
2. **Alternative hypothesis (H₁)**: There is an effect
3. Compute **test statistic**
4. Calculate **p-value**: Probability of observing this if H₀ is true
5. If p-value < α (e.g., 0.05), reject H₀

In [ ]:
# A/B test example
np.random.seed(42)

# Group A: Control (old version)
group_A = np.random.normal(100, 15, 100)

# Group B: Treatment (new version) - slightly better
group_B = np.random.normal(105, 15, 100)

# Perform t-test
t_stat, p_value = stats.ttest_ind(group_A, group_B)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Distributions
ax1.hist(group_A, bins=20, alpha=0.6, label='Group A (Control)', 
         color='blue', edgecolor='black')
ax1.hist(group_B, bins=20, alpha=0.6, label='Group B (Treatment)', 
         color='red', edgecolor='black')
ax1.axvline(np.mean(group_A), color='blue', linestyle='--', linewidth=2.5,
            label=f'Mean A = {np.mean(group_A):.1f}')
ax1.axvline(np.mean(group_B), color='red', linestyle='--', linewidth=2.5,
            label=f'Mean B = {np.mean(group_B):.1f}')
ax1.set_xlabel('Value', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('A/B Test: Comparing Two Groups', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# T-test visualization
x = np.linspace(-4, 4, 1000)
df = len(group_A) + len(group_B) - 2
t_dist = stats.t.pdf(x, df)
ax2.plot(x, t_dist, 'b-', linewidth=2.5, label='t-distribution')
ax2.fill_between(x, t_dist, where=(x >= t_stat), alpha=0.3, 
                 color='red', label=f'p-value = {p_value:.4f}')
ax2.axvline(t_stat, color='red', linestyle='--', linewidth=2.5, 
            label=f't-statistic = {t_stat:.2f}')
ax2.axvline(1.96, color='green', linestyle=':', linewidth=2, 
            label='Critical value (α=0.05)')
ax2.axvline(-1.96, color='green', linestyle=':', linewidth=2)
ax2.set_xlabel('t-statistic', fontsize=12)
ax2.set_ylabel('Probability Density', fontsize=12)
ax2.set_title('t-Test Result', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.suptitle('🔬 Hypothesis Testing: t-Test', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("=" * 60)
print("HYPOTHESIS TEST RESULTS")
print("=" * 60)
print(f"Group A mean: {np.mean(group_A):.2f} ± {np.std(group_A):.2f}")
print(f"Group B mean: {np.mean(group_B):.2f} ± {np.std(group_B):.2f}")
print(f"Difference: {np.mean(group_B) - np.mean(group_A):.2f}")
print(f"\nt-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")
print(f"\nConclusion: ", end="")
if p_value < 0.05:
    print("Reject null hypothesis - Groups ARE significantly different!")
else:
    print("Fail to reject null hypothesis - No significant difference.")

## 3. Confidence Intervals

Instead of a point estimate, give a range:

**95% Confidence Interval**: If we repeated this experiment 100 times, 95 of those intervals would contain the true parameter.

In [ ]:
# True population mean
true_mean = 50
true_std = 10

# Simulate many experiments
n_experiments = 50
sample_size = 30
confidence_level = 0.95

# Store results
means = []
confidence_intervals = []
contains_true_mean = []

for _ in range(n_experiments):
    # Sample from population
    sample = np.random.normal(true_mean, true_std, sample_size)
    
    # Compute confidence interval
    sample_mean = np.mean(sample)
    sem = stats.sem(sample)  # Standard error of mean
    ci = stats.t.interval(confidence_level, len(sample)-1, 
                          loc=sample_mean, scale=sem)
    
    means.append(sample_mean)
    confidence_intervals.append(ci)
    contains_true_mean.append(ci[0] <= true_mean <= ci[1])

# Visualize
fig, ax = plt.subplots(figsize=(14, 10))

for i, (mean, ci, contains) in enumerate(zip(means, confidence_intervals, 
                                               contains_true_mean)):
    color = 'green' if contains else 'red'
    alpha = 0.6 if contains else 1.0
    linewidth = 1.5 if contains else 2.5
    
    # Plot confidence interval
    ax.plot([ci[0], ci[1]], [i, i], color=color, linewidth=linewidth, alpha=alpha)
    # Plot point estimate
    ax.plot(mean, i, 'o', color=color, markersize=6, alpha=alpha)

# Mark true mean
ax.axvline(x=true_mean, color='blue', linestyle='--', linewidth=3, 
           label=f'True Mean = {true_mean}')

ax.set_xlabel('Value', fontsize=12)
ax.set_ylabel('Experiment Number', fontsize=12)
ax.set_title(f'{confidence_level:.0%} Confidence Intervals from {n_experiments} Experiments', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=12, loc='upper right')
ax.grid(True, alpha=0.3, axis='x')

# Add caption
coverage = np.mean(contains_true_mean) * 100
caption = f'Green: Contains true mean ({np.sum(contains_true_mean)}/{n_experiments})\n'
caption += f'Red: Misses true mean ({n_experiments - np.sum(contains_true_mean)}/{n_experiments})\n'
caption += f'Coverage: {coverage:.1f}%'
ax.text(0.02, 0.98, caption, transform=ax.transAxes, 
        fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.tight_layout()
plt.show()

print(f"Expected to contain true mean: {confidence_level:.0%}")
print(f"Actual coverage: {coverage:.1f}%")

## 4. Correlation vs Causation

**Correlation**: Two variables move together

**Causation**: One variable causes changes in another

⚠️ **Correlation ≠ Causation!**

In [ ]:
# Generate examples
n = 100
x = np.linspace(0, 10, n)

# Scenario 1: Causal relationship
y_causal = 2 * x + np.random.normal(0, 1, n)

# Scenario 2: Spurious correlation (both caused by third variable)
z = np.linspace(0, 10, n)  # Hidden variable (e.g., time)
a_spurious = z + np.random.normal(0, 0.5, n)
b_spurious = z + np.random.normal(0, 0.5, n)

# Scenario 3: No correlation
x_random = np.random.normal(5, 2, n)
y_random = np.random.normal(5, 2, n)

# Compute correlations
corr_causal = np.corrcoef(x, y_causal)[0, 1]
corr_spurious = np.corrcoef(a_spurious, b_spurious)[0, 1]
corr_random = np.corrcoef(x_random, y_random)[0, 1]

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Causal
axes[0].scatter(x, y_causal, alpha=0.6, s=50, color='green')
axes[0].plot(x, 2*x, 'r--', linewidth=2, label='True relationship')
axes[0].set_xlabel('X (cause)', fontsize=11)
axes[0].set_ylabel('Y (effect)', fontsize=11)
axes[0].set_title(f'Causal Relationship\nr = {corr_causal:.3f}', 
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Spurious
axes[1].scatter(a_spurious, b_spurious, alpha=0.6, s=50, color='orange')
axes[1].set_xlabel('A (ice cream sales)', fontsize=11)
axes[1].set_ylabel('B (shark attacks)', fontsize=11)
axes[1].set_title(f'Spurious Correlation\nr = {corr_spurious:.3f}\n(Both caused by summer)', 
                  fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# No correlation
axes[2].scatter(x_random, y_random, alpha=0.6, s=50, color='blue')
axes[2].set_xlabel('X', fontsize=11)
axes[2].set_ylabel('Y', fontsize=11)
axes[2].set_title(f'No Correlation\nr = {corr_random:.3f}', 
                  fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.suptitle('⚠️ Correlation vs Causation', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Key lesson: High correlation doesn't prove causation!")
print("Always consider: confounding variables, reverse causation, coincidence.")

## 5. Bias-Variance Tradeoff

**Bias**: Error from wrong assumptions (underfitting)

**Variance**: Error from sensitivity to training data (overfitting)

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$$

In [ ]:
# True function
def true_function(x):
    return np.sin(x) + 0.5 * x

# Generate data
np.random.seed(42)
X_train = np.linspace(0, 10, 20)
y_train = true_function(X_train) + np.random.normal(0, 0.5, len(X_train))
X_test = np.linspace(0, 10, 200)
y_true = true_function(X_test)

# Fit models of different complexity
degrees = [1, 3, 15]
titles = ['High Bias (Underfitting)', 'Just Right', 'High Variance (Overfitting)']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, degree, title in zip(axes, degrees, titles):
    # Fit polynomial
    coeffs = np.polyfit(X_train, y_train, degree)
    y_pred = np.polyval(coeffs, X_test)
    
    # Plot
    ax.plot(X_test, y_true, 'g-', linewidth=3, label='True function', alpha=0.7)
    ax.scatter(X_train, y_train, s=80, c='blue', alpha=0.6, 
               edgecolors='black', linewidths=1.5, label='Training data', zorder=5)
    ax.plot(X_test, y_pred, 'r--', linewidth=2.5, label=f'Model (degree {degree})')
    
    # Calculate errors
    train_pred = np.polyval(coeffs, X_train)
    train_mse = np.mean((y_train - train_pred)**2)
    test_mse = np.mean((y_true - y_pred)**2)
    
    ax.set_xlabel('X', fontsize=11)
    ax.set_ylabel('Y', fontsize=11)
    ax.set_title(f'{title}\nTrain MSE: {train_mse:.3f}, Test MSE: {test_mse:.3f}', 
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-2, 8)

plt.suptitle('⚖️ Bias-Variance Tradeoff', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Left: High bias - model too simple")
print("Middle: Good balance")
print("Right: High variance - model memorizes noise")

## 🚀 Key Takeaways

1. **Descriptive statistics** summarize data
2. **Hypothesis testing** determines if effects are real
3. **Confidence intervals** quantify uncertainty
4. **Correlation ≠ causation** - be careful!
5. **Bias-variance tradeoff** is fundamental to ML
6. **p-values** can be misleading - interpret carefully

## Applications in ML

- Model evaluation and comparison
- A/B testing for model deployment
- Feature selection and importance
- Uncertainty quantification
- Detecting overfitting/underfitting

## Resources
- "Statistics for Data Science" by James, Witten, Hastie, Tibshirani
- "Practical Statistics for Data Scientists" by Bruce & Bruce
- Khan Academy: Statistics & Probability